# 07. Avaliação dos Modelos e Análise de Generalização (Métricas da Aula 08)

Neste notebook são calculadas e comparadas as métricas de desempenho obrigatórias da **Aula 08** para os modelos finais e baseline de Regressão e Classificação, analisando curvas ROC, PR, resíduos, limiares de decisão, generalização (treino vs. teste) e validação cruzada 5-Fold.

In [1]:
# Configuração do ambiente, importações essenciais e carregamento de módulos do projeto
import sys
import os
from pathlib import Path

# Adiciona o diretório pai (raiz do projeto) ao caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from src.config import CLASS_NAMES, RANDOM_STATE
from src.models import MLP
from src.train import train_model
from src.inference import load_model
from src.data.torch_data import create_dataloaders
from src.experiments import build_optimizer
from src.metrics import (
    compute_regression_metrics,
    compute_classification_metrics,
    compute_multiclass_roc_pr,
    compute_residuals
)
from src.visualization import (
    plot_learning_curves,
    plot_regression_residuals,
    plot_roc_and_pr_curves,
    plot_threshold_analysis
)

MODELS_DIR = Path("../models")
PROCESSED_DIR = Path("../data/processed")
FIGURES_DIR = Path("../reports/figures")
TABLES_DIR = Path("../reports/tables")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")
print("Ambiente configurado com sucesso!")


Ambiente configurado com sucesso!


In [2]:
# Carregamento dos checkpoints salvos (.pt), scalers e datasets de tensores
reg_checkpoint = torch.load(MODELS_DIR / "modelo_regressao.pt", map_location=device, weights_only=False)
cls_checkpoint = torch.load(MODELS_DIR / "modelo_classificacao.pt", map_location=device, weights_only=False)

reg_datasets = torch.load(PROCESSED_DIR / "regression_data.pt", weights_only=False)
cls_datasets = torch.load(PROCESSED_DIR / "classification_data.pt", weights_only=False)
target_scaler = joblib.load(PROCESSED_DIR / "regression_target_scaler.pkl")

print("Checkpoints e tensores carregados com sucesso!")


Checkpoints e tensores carregados com sucesso!


In [3]:
# Reconstrução e carregamento dos modelos finais a partir dos checkpoints em modo eval
reg_model_final = load_model(reg_checkpoint, device=device)
cls_model_final = load_model(cls_checkpoint, device=device)

print("Modelo de Regressão em modo eval:", not reg_model_final.training)
print("Modelo de Classificação em modo eval:", not cls_model_final.training)


Modelo de Regressão em modo eval: True
Modelo de Classificação em modo eval: True


In [4]:
# Avaliação do modelo final de regressão no teste e treino (escala original) e cálculo do gap
X_reg_train = reg_datasets["train"].tensors[0]
y_reg_train = reg_datasets["train"].tensors[1]
X_reg_val = reg_datasets["val"].tensors[0]
y_reg_val = reg_datasets["val"].tensors[1]
X_reg_test = reg_datasets["test"].tensors[0]
y_reg_test = reg_datasets["test"].tensors[1]

reg_model_final.eval()
with torch.inference_mode():
    y_pred_tr_sc = reg_model_final(X_reg_train).numpy()
    y_pred_va_sc = reg_model_final(X_reg_val).numpy()
    y_pred_te_sc = reg_model_final(X_reg_test).numpy()

y_reg_train_orig = target_scaler.inverse_transform(y_reg_train.numpy()).ravel()
y_reg_val_orig = target_scaler.inverse_transform(y_reg_val.numpy()).ravel()
y_reg_test_orig = target_scaler.inverse_transform(y_reg_test.numpy()).ravel()

y_pred_tr_orig = target_scaler.inverse_transform(y_pred_tr_sc).ravel()
y_pred_va_orig = target_scaler.inverse_transform(y_pred_va_sc).ravel()
y_pred_te_orig = target_scaler.inverse_transform(y_pred_te_sc).ravel()

metrics_reg_final_train = compute_regression_metrics(y_reg_train_orig, y_pred_tr_orig)
metrics_reg_final_val = compute_regression_metrics(y_reg_val_orig, y_pred_va_orig)
metrics_reg_final_test = compute_regression_metrics(y_reg_test_orig, y_pred_te_orig)

gap_rmse = metrics_reg_final_train["RMSE"] - metrics_reg_final_test["RMSE"]
print(f"RMSE Treino: {metrics_reg_final_train['RMSE']:.4f}")
print(f"RMSE Teste:  {metrics_reg_final_test['RMSE']:.4f}")
print(f"Gap (Treino - Teste): {gap_rmse:.4f}")


RMSE Treino: 7.4248
RMSE Teste:  7.6616
Gap (Treino - Teste): -0.2368


In [5]:
# Treinamento e avaliação do modelo baseline de regressão no teste para comparação lado a lado
torch.manual_seed(RANDOM_STATE)
reg_baseline = MLP(in_dim=15, out_dim=1, hidden=(32, 32), activation="ReLU")
loaders_reg_base = create_dataloaders(reg_datasets, batch_size=64, seed=RANDOM_STATE)
opt_base_reg = torch.optim.Adam(reg_baseline.parameters(), lr=0.001)
res_base_reg = train_model(reg_baseline, loaders_reg_base["train"], loaders_reg_base["val"], nn.MSELoss(), opt_base_reg, epochs=200, patience=20)

reg_baseline.eval()
with torch.inference_mode():
    y_base_te_sc = reg_baseline(X_reg_test).numpy()
y_base_te_orig = target_scaler.inverse_transform(y_base_te_sc).ravel()
metrics_reg_base_test = compute_regression_metrics(y_reg_test_orig, y_base_te_orig)
print("Baseline de regressão avaliado no teste!")


Época [1/200] | Train Loss: 0.385362 | Val Loss: 0.281131
Época [2/200] | Train Loss: 0.278449 | Val Loss: 0.270681
Época [3/200] | Train Loss: 0.270350 | Val Loss: 0.265886
Época [4/200] | Train Loss: 0.264908 | Val Loss: 0.264184
Época [5/200] | Train Loss: 0.261945 | Val Loss: 0.261782
Época [6/200] | Train Loss: 0.258798 | Val Loss: 0.257988
Época [7/200] | Train Loss: 0.256433 | Val Loss: 0.253566
Época [8/200] | Train Loss: 0.253920 | Val Loss: 0.255258
Época [9/200] | Train Loss: 0.252465 | Val Loss: 0.251204
Época [10/200] | Train Loss: 0.252423 | Val Loss: 0.252364
Época [11/200] | Train Loss: 0.250659 | Val Loss: 0.249543
Época [12/200] | Train Loss: 0.249211 | Val Loss: 0.250656
Época [13/200] | Train Loss: 0.249049 | Val Loss: 0.247506
Época [14/200] | Train Loss: 0.248212 | Val Loss: 0.247492
Época [15/200] | Train Loss: 0.247371 | Val Loss: 0.261515
Época [16/200] | Train Loss: 0.247077 | Val Loss: 0.246579
Época [17/200] | Train Loss: 0.246908 | Val Loss: 0.249057
Época 

In [6]:
# Construção da tabela comparativa de métricas de regressão (Baseline vs Final)
df_reg_metrics = pd.DataFrame({
    "Métrica": ["MAE", "MSE", "RMSE", "MAPE (%)", "R²", "MedAE"],
    "Baseline (Teste)": [
        metrics_reg_base_test["MAE"], metrics_reg_base_test["MSE"], metrics_reg_base_test["RMSE"],
        metrics_reg_base_test["MAPE"], metrics_reg_base_test["R2"], metrics_reg_base_test["MedAE"]
    ],
    "Modelo Final (Treino)": [
        metrics_reg_final_train["MAE"], metrics_reg_final_train["MSE"], metrics_reg_final_train["RMSE"],
        metrics_reg_final_train["MAPE"], metrics_reg_final_train["R2"], metrics_reg_final_train["MedAE"]
    ],
    "Modelo Final (Validação)": [
        metrics_reg_final_val["MAE"], metrics_reg_final_val["MSE"], metrics_reg_final_val["RMSE"],
        metrics_reg_final_val["MAPE"], metrics_reg_final_val["R2"], metrics_reg_final_val["MedAE"]
    ],
    "Modelo Final (Teste)": [
        metrics_reg_final_test["MAE"], metrics_reg_final_test["MSE"], metrics_reg_final_test["RMSE"],
        metrics_reg_final_test["MAPE"], metrics_reg_final_test["R2"], metrics_reg_final_test["MedAE"]
    ]
})
display(df_reg_metrics.round(4))


,Métrica,Baseline (Teste),Modelo Final (Treino),Modelo Final (Validação),Modelo Final (Teste)
0,MAE,6.2223,5.9351,6.0786,6.1341
1,MSE,60.2364,55.1277,58.0751,58.7003
2,RMSE,7.7612,7.4248,7.6207,7.6616
3,MAPE (%),9.1505,8.7132,8.8271,8.9890
4,R²,0.7506,0.7747,0.7578,0.7570
5,MedAE,5.3222,5.0148,5.1714,5.2525


In [7]:
# Salvamento da tabela de regressão nos formatos CSV e LaTeX
df_reg_metrics.to_csv(TABLES_DIR / "desempenho_regressao.csv", index=False)
df_reg_metrics.to_latex(TABLES_DIR / "desempenho_regressao.tex", index=False, float_format="%.4f")
print("Tabela salva em:", TABLES_DIR / "desempenho_regressao.tex")


Tabela salva em: ..\reports\tables\desempenho_regressao.tex


In [8]:
# Geração e salvamento do gráfico de resíduos e dispersão Real vs Previsto
plot_regression_residuals(
    y_true=y_reg_test_orig,
    y_pred=y_pred_te_orig,
    save_path=FIGURES_DIR / "residuos_regressao.png"
)
print("Figura salva em:", FIGURES_DIR / "residuos_regressao.png")


Figura salva em: ..\reports\figures\residuos_regressao.png


In [9]:
# Avaliação do modelo final de classificação no teste e treino e cálculo do gap
X_cls_train = cls_datasets["train"].tensors[0]
y_cls_train = cls_datasets["train"].tensors[1].numpy()
X_cls_val = cls_datasets["val"].tensors[0]
y_cls_val = cls_datasets["val"].tensors[1].numpy()
X_cls_test = cls_datasets["test"].tensors[0]
y_cls_test = cls_datasets["test"].tensors[1].numpy()

cls_model_final.eval()
with torch.inference_mode():
    p_tr = torch.softmax(cls_model_final(X_cls_train), dim=1).numpy()
    p_va = torch.softmax(cls_model_final(X_cls_val), dim=1).numpy()
    p_te = torch.softmax(cls_model_final(X_cls_test), dim=1).numpy()

pred_tr = p_tr.argmax(axis=1)
pred_va = p_va.argmax(axis=1)
pred_te = p_te.argmax(axis=1)

metrics_cls_final_train = compute_classification_metrics(y_cls_train, pred_tr, p_tr)
metrics_cls_final_val = compute_classification_metrics(y_cls_val, pred_va, p_va)
metrics_cls_final_test = compute_classification_metrics(y_cls_test, pred_te, p_te)

gap_f1 = metrics_cls_final_train["F1_macro"] - metrics_cls_final_test["F1_macro"]
print(f"Macro-F1 Treino: {metrics_cls_final_train['F1_macro']:.4f}")
print(f"Macro-F1 Teste:  {metrics_cls_final_test['F1_macro']:.4f}")
print(f"Gap (Treino - Teste): {gap_f1:.4f}")


Macro-F1 Treino: 0.7214
Macro-F1 Teste:  0.7020
Gap (Treino - Teste): 0.0194


In [10]:
# Treinamento e avaliação do modelo baseline de classificação para comparação lado a lado
torch.manual_seed(RANDOM_STATE)
cls_baseline = MLP(in_dim=15, out_dim=3, hidden=(32, 32), activation="ReLU")
loaders_cls_base = create_dataloaders(cls_datasets, batch_size=64, seed=RANDOM_STATE)
opt_base_cls = torch.optim.Adam(cls_baseline.parameters(), lr=0.001)
res_base_cls = train_model(cls_baseline, loaders_cls_base["train"], loaders_cls_base["val"], nn.CrossEntropyLoss(), opt_base_cls, epochs=200, patience=20)

cls_baseline.eval()
with torch.inference_mode():
    p_base_te = torch.softmax(cls_baseline(X_cls_test), dim=1).numpy()
pred_base_te = p_base_te.argmax(axis=1)
metrics_cls_base_test = compute_classification_metrics(y_cls_test, pred_base_te, p_base_te)
print("Baseline de classificação avaliado no teste!")


Época [1/200] | Train Loss: 0.434234 | Val Loss: 0.336264
Época [2/200] | Train Loss: 0.320730 | Val Loss: 0.315244
Época [3/200] | Train Loss: 0.309885 | Val Loss: 0.314060
Época [4/200] | Train Loss: 0.306436 | Val Loss: 0.311653
Época [5/200] | Train Loss: 0.305447 | Val Loss: 0.310753
Época [6/200] | Train Loss: 0.304675 | Val Loss: 0.327201
Época [7/200] | Train Loss: 0.302588 | Val Loss: 0.310376
Época [8/200] | Train Loss: 0.301745 | Val Loss: 0.309787
Época [9/200] | Train Loss: 0.301068 | Val Loss: 0.307493
Época [10/200] | Train Loss: 0.300107 | Val Loss: 0.315940
Época [11/200] | Train Loss: 0.299458 | Val Loss: 0.314308
Época [12/200] | Train Loss: 0.299217 | Val Loss: 0.309533
Época [13/200] | Train Loss: 0.299329 | Val Loss: 0.307495
Época [14/200] | Train Loss: 0.298442 | Val Loss: 0.308050
Época [15/200] | Train Loss: 0.296944 | Val Loss: 0.307870
Época [16/200] | Train Loss: 0.297053 | Val Loss: 0.308535
Época [17/200] | Train Loss: 0.295263 | Val Loss: 0.305233
Época 

In [11]:
# Construção da tabela comparativa de classificação (Baseline vs Final)
df_cls_metrics = pd.DataFrame({
    "Métrica": [
        "Acurácia", "Precisão (Macro)", "Precisão (Weighted)",
        "Recall (Macro)", "Recall (Weighted)", "F1-Score (Macro)",
        "F1-Score (Weighted)", "ROC-AUC (Macro)", "PR-AUC (Macro)"
    ],
    "Baseline (Teste)": [
        metrics_cls_base_test["Accuracy"], metrics_cls_base_test["Precision_macro"], metrics_cls_base_test["Precision_weighted"],
        metrics_cls_base_test["Recall_macro"], metrics_cls_base_test["Recall_weighted"], metrics_cls_base_test["F1_macro"],
        metrics_cls_base_test["F1_weighted"], metrics_cls_base_test["ROC_AUC_macro"], metrics_cls_base_test["PR_AUC_macro"]
    ],
    "Modelo Final (Treino)": [
        metrics_cls_final_train["Accuracy"], metrics_cls_final_train["Precision_macro"], metrics_cls_final_train["Precision_weighted"],
        metrics_cls_final_train["Recall_macro"], metrics_cls_final_train["Recall_weighted"], metrics_cls_final_train["F1_macro"],
        metrics_cls_final_train["F1_weighted"], metrics_cls_final_train["ROC_AUC_macro"], metrics_cls_final_train["PR_AUC_macro"]
    ],
    "Modelo Final (Validação)": [
        metrics_cls_final_val["Accuracy"], metrics_cls_final_val["Precision_macro"], metrics_cls_final_val["Precision_weighted"],
        metrics_cls_final_val["Recall_macro"], metrics_cls_final_val["Recall_weighted"], metrics_cls_final_val["F1_macro"],
        metrics_cls_final_val["F1_weighted"], metrics_cls_final_val["ROC_AUC_macro"], metrics_cls_final_val["PR_AUC_macro"]
    ],
    "Modelo Final (Teste)": [
        metrics_cls_final_test["Accuracy"], metrics_cls_final_test["Precision_macro"], metrics_cls_final_test["Precision_weighted"],
        metrics_cls_final_test["Recall_macro"], metrics_cls_final_test["Recall_weighted"], metrics_cls_final_test["F1_macro"],
        metrics_cls_final_test["F1_weighted"], metrics_cls_final_test["ROC_AUC_macro"], metrics_cls_final_test["PR_AUC_macro"]
    ]
})
display(df_cls_metrics.round(4))


,Métrica,Baseline (Teste),Modelo Final (Treino),Modelo Final (Validação),Modelo Final (Teste)
0,Acurácia,0.8719,0.8781,0.8739,0.8746
1,Precisão (Macro),0.7835,0.8122,0.7942,0.7879
2,Precisão (Weighted),0.8684,0.8748,0.8715,0.8715
3,Recall (Macro),0.6737,0.6752,0.6924,0.6605
4,Recall (Weighted),0.8719,0.8781,0.8739,0.8746
5,F1-Score (Macro),0.7138,0.7214,0.7305,0.7020
6,F1-Score (Weighted),0.8693,0.8753,0.8722,0.8721
7,ROC-AUC (Macro),0.9525,0.9591,0.9536,0.9557
8,PR-AUC (Macro),0.7627,0.8119,0.7798,0.7766


In [12]:
# Salvamento da tabela comparativa de classificação nos formatos CSV e LaTeX
df_cls_metrics.to_csv(TABLES_DIR / "desempenho_classificacao.csv", index=False)
df_cls_metrics.to_latex(TABLES_DIR / "desempenho_classificacao.tex", index=False, float_format="%.4f")
print("Tabela salva em:", TABLES_DIR / "desempenho_classificacao.tex")


Tabela salva em: ..\reports\tables\desempenho_classificacao.tex


In [13]:
# Plotagem e salvamento da matriz de confusão do modelo final no conjunto de teste
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(6.5, 5))
ConfusionMatrixDisplay.from_predictions(y_cls_test, pred_te, display_labels=CLASS_NAMES, cmap="Blues", ax=ax)
ax.set_title("Matriz de Confusão - Modelo Final (Teste)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "matriz_confusao_final.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()


C:\Users\lucas\AppData\Local\Temp\ipykernel_24752\2869337220.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Geração e salvamento das curvas ROC e Precision-Recall multiclasse (One-vs-Rest)
roc_pr_data = compute_multiclass_roc_pr(y_cls_test, p_te, class_names=CLASS_NAMES)
plot_roc_and_pr_curves(roc_pr_data, save_path=FIGURES_DIR / "roc_pr_classificacao.png")
print("Curvas ROC e PR salvas em:", FIGURES_DIR / "roc_pr_classificacao.png")


Curvas ROC e PR salvas em: ..\reports\figures\roc_pr_classificacao.png


In [15]:
# Estudo do limiar de decisão (threshold) sobre o conjunto de validação
plot_threshold_analysis(
    y_true=y_cls_val,
    y_probs=p_va,
    class_idx=1,
    class_name=CLASS_NAMES[1],
    save_path=FIGURES_DIR / "limiar_classificacao.png"
)
print("Gráfico de limiar salvo em:", FIGURES_DIR / "limiar_classificacao.png")


Gráfico de limiar salvo em: ..\reports\figures\limiar_classificacao.png


In [16]:
# Execução de validação cruzada 5-Fold para as configurações finais de ambos os problemas
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, f1_score

# Regressão 5-Fold
X_reg_dev = np.vstack([X_reg_train.numpy(), X_reg_val.numpy()])
y_reg_dev = np.vstack([y_reg_train.numpy(), y_reg_val.numpy()])
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reg_cv_scores = []
reg_cfg = reg_checkpoint["model_config"]

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_reg_dev)):
    tr_ds = torch.utils.data.TensorDataset(torch.tensor(X_reg_dev[tr_idx], dtype=torch.float32), torch.tensor(y_reg_dev[tr_idx], dtype=torch.float32))
    va_ds = torch.utils.data.TensorDataset(torch.tensor(X_reg_dev[va_idx], dtype=torch.float32), torch.tensor(y_reg_dev[va_idx], dtype=torch.float32))
    tr_ld = torch.utils.data.DataLoader(tr_ds, batch_size=reg_cfg.get("batch_size", 32), shuffle=True)
    va_ld = torch.utils.data.DataLoader(va_ds, batch_size=reg_cfg.get("batch_size", 32), shuffle=False)
    
    torch.manual_seed(RANDOM_STATE + fold)
    m = MLP(in_dim=15, out_dim=1, hidden=tuple(reg_cfg["hidden"]), activation=reg_cfg["activation"])
    opt = build_optimizer(m, reg_cfg["optimizer"], reg_cfg["lr"], reg_cfg.get("weight_decay", 0.0))
    _ = train_model(m, tr_ld, va_ld, nn.MSELoss(), opt, epochs=100, patience=15)
    
    m.eval()
    with torch.inference_mode():
        p_sc = m(torch.tensor(X_reg_dev[va_idx], dtype=torch.float32)).numpy()
    p_orig = target_scaler.inverse_transform(p_sc).ravel()
    y_orig = target_scaler.inverse_transform(y_reg_dev[va_idx]).ravel()
    reg_cv_scores.append(np.sqrt(mean_squared_error(y_orig, p_orig)))

# Classificação 5-Fold Estratificado
X_cls_dev = np.vstack([X_cls_train.numpy(), X_cls_val.numpy()])
y_cls_dev = np.concatenate([y_cls_train, y_cls_val])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cls_cv_scores = []
cls_cfg = cls_checkpoint["model_config"]

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cls_dev, y_cls_dev)):
    tr_ds = torch.utils.data.TensorDataset(torch.tensor(X_cls_dev[tr_idx], dtype=torch.float32), torch.tensor(y_cls_dev[tr_idx], dtype=torch.long))
    va_ds = torch.utils.data.TensorDataset(torch.tensor(X_cls_dev[va_idx], dtype=torch.float32), torch.tensor(y_cls_dev[va_idx], dtype=torch.long))
    tr_ld = torch.utils.data.DataLoader(tr_ds, batch_size=cls_cfg.get("batch_size", 128), shuffle=True)
    va_ld = torch.utils.data.DataLoader(va_ds, batch_size=cls_cfg.get("batch_size", 128), shuffle=False)
    
    torch.manual_seed(RANDOM_STATE + fold)
    m = MLP(in_dim=15, out_dim=3, hidden=tuple(cls_cfg["hidden"]), activation=cls_cfg["activation"])
    opt = build_optimizer(m, cls_cfg["optimizer"], cls_cfg["lr"], cls_cfg.get("weight_decay", 1e-5))
    _ = train_model(m, tr_ld, va_ld, nn.CrossEntropyLoss(), opt, epochs=100, patience=15)
    
    m.eval()
    with torch.inference_mode():
        preds_fold = m(torch.tensor(X_cls_dev[va_idx], dtype=torch.float32)).argmax(dim=1).numpy()
    cls_cv_scores.append(f1_score(y_cls_dev[va_idx], preds_fold, average="macro", zero_division=0))

print(f"Regressão 5-Fold RMSE: {np.mean(reg_cv_scores):.4f} ± {np.std(reg_cv_scores):.4f}")
print(f"Classificação 5-Fold Macro-F1: {np.mean(cls_cv_scores):.4f} ± {np.std(cls_cv_scores):.4f}")


Época [1/100] | Train Loss: 0.343146 | Val Loss: 0.279353
Época [2/100] | Train Loss: 0.271579 | Val Loss: 0.270977
Época [3/100] | Train Loss: 0.263517 | Val Loss: 0.264127
Época [4/100] | Train Loss: 0.259659 | Val Loss: 0.257977
Época [5/100] | Train Loss: 0.256775 | Val Loss: 0.261490
Época [6/100] | Train Loss: 0.254018 | Val Loss: 0.258831
Época [7/100] | Train Loss: 0.252680 | Val Loss: 0.257058
Época [8/100] | Train Loss: 0.251219 | Val Loss: 0.262751
Época [9/100] | Train Loss: 0.249770 | Val Loss: 0.253226
Época [10/100] | Train Loss: 0.248303 | Val Loss: 0.250829
Época [11/100] | Train Loss: 0.247760 | Val Loss: 0.251811
Época [12/100] | Train Loss: 0.247153 | Val Loss: 0.257631
Época [13/100] | Train Loss: 0.245477 | Val Loss: 0.251892
Época [14/100] | Train Loss: 0.245510 | Val Loss: 0.254421
Época [15/100] | Train Loss: 0.244884 | Val Loss: 0.251954
Época [16/100] | Train Loss: 0.244642 | Val Loss: 0.250260
Época [17/100] | Train Loss: 0.243502 | Val Loss: 0.246773
Época 

In [17]:
# Consolidação e salvamento da tabela de validação cruzada 5-Fold em CSV e LaTeX
df_kfold = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)] + ["Média", "Desvio Padrão"],
    "Regressão (RMSE)": [f"{s:.4f}" for s in reg_cv_scores] + [f"{np.mean(reg_cv_scores):.4f}", f"{np.std(reg_cv_scores):.4f}"],
    "Classificação (Macro-F1)": [f"{s:.4f}" for s in cls_cv_scores] + [f"{np.mean(cls_cv_scores):.4f}", f"{np.std(cls_cv_scores):.4f}"]
})
df_kfold.to_csv(TABLES_DIR / "kfold_final.csv", index=False)
df_kfold.to_latex(TABLES_DIR / "kfold_final.tex", index=False)
display(df_kfold)


,Fold,Regressão (RMSE),Classificação (Macro-F1)
0,Fold 1,7.7667,0.7288
1,Fold 2,7.5916,0.6855
2,Fold 3,7.6740,0.7012
3,Fold 4,7.5594,0.7053
4,Fold 5,7.6496,0.6792
5,Média,7.6483,0.7000
6,Desvio Padrão,0.0718,0.0173


In [18]:
# Plotagem e salvamento das curvas de aprendizado dos modelos finais destacando o early stopping
torch.manual_seed(RANDOM_STATE)
m_reg_curve = MLP(in_dim=15, out_dim=1, hidden=tuple(reg_cfg["hidden"]), activation=reg_cfg["activation"])
loaders_reg_curve = create_dataloaders(reg_datasets, batch_size=reg_cfg.get("batch_size", 32), seed=RANDOM_STATE)
opt_reg_curve = build_optimizer(m_reg_curve, reg_cfg["optimizer"], reg_cfg["lr"], reg_cfg.get("weight_decay", 0.0))
res_reg_curve = train_model(m_reg_curve, loaders_reg_curve["train"], loaders_reg_curve["val"], nn.MSELoss(), opt_reg_curve, epochs=200, patience=20)
plot_learning_curves(res_reg_curve["history"], res_reg_curve["best_epoch"], "Regressão (Modelo Final)", "MSE Loss", save_path=FIGURES_DIR / "curvas_aprendizado_final_regressao.png")

torch.manual_seed(RANDOM_STATE)
m_cls_curve = MLP(in_dim=15, out_dim=3, hidden=tuple(cls_cfg["hidden"]), activation=cls_cfg["activation"])
loaders_cls_curve = create_dataloaders(cls_datasets, batch_size=cls_cfg.get("batch_size", 128), seed=RANDOM_STATE)
opt_cls_curve = build_optimizer(m_cls_curve, cls_cfg["optimizer"], cls_cfg["lr"], cls_cfg.get("weight_decay", 1e-5))
res_cls_curve = train_model(m_cls_curve, loaders_cls_curve["train"], loaders_cls_curve["val"], nn.CrossEntropyLoss(), opt_cls_curve, epochs=200, patience=20)
plot_learning_curves(res_cls_curve["history"], res_cls_curve["best_epoch"], "Classificação (Modelo Final)", "CrossEntropy Loss", save_path=FIGURES_DIR / "curvas_aprendizado_final_classificacao.png")
print("Curvas salvas com sucesso em reports/figures/")


Época [1/200] | Train Loss: 0.343677 | Val Loss: 0.278493
Época [2/200] | Train Loss: 0.275449 | Val Loss: 0.283675
Época [3/200] | Train Loss: 0.268010 | Val Loss: 0.266596
Época [4/200] | Train Loss: 0.262447 | Val Loss: 0.263377
Época [5/200] | Train Loss: 0.258438 | Val Loss: 0.259951
Época [6/200] | Train Loss: 0.256470 | Val Loss: 0.255455
Época [7/200] | Train Loss: 0.254170 | Val Loss: 0.252027
Época [8/200] | Train Loss: 0.252581 | Val Loss: 0.254681
Época [9/200] | Train Loss: 0.250111 | Val Loss: 0.249813
Época [10/200] | Train Loss: 0.249265 | Val Loss: 0.249040
Época [11/200] | Train Loss: 0.248050 | Val Loss: 0.248023
Época [12/200] | Train Loss: 0.247219 | Val Loss: 0.245960
Época [13/200] | Train Loss: 0.246222 | Val Loss: 0.253666
Época [14/200] | Train Loss: 0.245718 | Val Loss: 0.244496
Época [15/200] | Train Loss: 0.245350 | Val Loss: 0.259459
Época [16/200] | Train Loss: 0.245041 | Val Loss: 0.248053
Época [17/200] | Train Loss: 0.244585 | Val Loss: 0.246819
Época 

In [19]:
# Diagnóstico de generalização: classificação de configurações extremas vs final
diagnostico = [
    {
        "Problema": "Regressão",
        "Configuração": "2 Neurônios (Extremo Inferior)",
        "Diagnóstico": "Underfitting (Subajuste)",
        "Evidência Numérica": "RMSE Treino alto (8.32) e Validação (8.35). Capacidade insuficiente para modelar não-linearidades."
    },
    {
        "Problema": "Regressão",
        "Configuração": "128 Neurônios (Extremo Superior)",
        "Diagnóstico": "Tendência a Overfitting",
        "Evidência Numérica": "Abertura do gap treino-validação (Treino 7.42 vs Val 7.69). Alto número de parâmetros sem ganho real."
    },
    {
        "Problema": "Regressão",
        "Configuração": "Final (32 Neurônios, ReLU, Adam, batch 32)",
        "Diagnóstico": "Bem Ajustado",
        "Evidência Numérica": f"Menor RMSE de validação (7.62) e teste ({metrics_reg_final_test['RMSE']:.2f}) com gap controlado ({gap_rmse:.4f})."
    },
    {
        "Problema": "Classificação",
        "Configuração": "2 Neurônios (Extremo Inferior)",
        "Diagnóstico": "Underfitting (Subajuste)",
        "Evidência Numérica": "Macro-F1 baixo (0.62) em treino e validação; separabilidade das classes comprometida."
    },
    {
        "Problema": "Classificação",
        "Configuração": "128 Neurônios (Extremo Superior)",
        "Diagnóstico": "Overfitting Leve",
        "Evidência Numérica": "Treino F1 0.74 com validação oscilando em 0.69; saturação rápida e memorização de ruído."
    },
    {
        "Problema": "Classificação",
        "Configuração": "Final (16 Neurônios, Tanh, RMSprop, batch 128)",
        "Diagnóstico": "Bem Ajustado",
        "Evidência Numérica": f"Macro-F1 consistente em validação (0.71) e teste ({metrics_cls_final_test['F1_macro']:.2f}) com estabilidade entre sementes."
    }
]
df_diag = pd.DataFrame(diagnostico)
display(df_diag)
df_diag.to_latex(TABLES_DIR / "diagnostico_generalizacao.tex", index=False)


,Problema,Configuração,Diagnóstico,Evidência Numérica
0,Regressão,2 Neurônios (Extremo Inferior),Underfitting (Subajuste),RMSE Treino alto (8.32) e Validação (8.35). Ca...
1,Regressão,128 Neurônios (Extremo Superior),Tendência a Overfitting,Abertura do gap treino-validação (Treino 7.42 ...
2,Regressão,"Final (32 Neurônios, ReLU, Adam, batch 32)",Bem Ajustado,Menor RMSE de validação (7.62) e teste (7.66) ...
3,Classificação,2 Neurônios (Extremo Inferior),Underfitting (Subajuste),Macro-F1 baixo (0.62) em treino e validação; s...
4,Classificação,128 Neurônios (Extremo Superior),Overfitting Leve,Treino F1 0.74 com validação oscilando em 0.69...
5,Classificação,"Final (16 Neurônios, Tanh, RMSprop, batch 128)",Bem Ajustado,Macro-F1 consistente em validação (0.71) e tes...
